# 11. LLM-Assisted Secure Code Review

## Background

Traditional code review catches ~30% of security vulnerabilities — human reviewers get fatigued, skip boilerplate, and miss subtle logic flaws. LLMs bring a different capability profile: tireless, consistent, context-aware review that can reason about security semantics rather than just matching patterns. But LLMs also hallucinate false positives and miss context-specific vulnerabilities that require business domain knowledge.

The industry is converging on a **hybrid model**: SAST for deterministic pattern matching, LLMs for semantic and logic-level analysis, humans for final judgment on high-severity findings. GitHub Copilot, Snyk Code, and Amazon CodeGuru all use LLMs in their review pipeline.

## What You'll Learn

- Structuring security-focused prompts for code review
- Building a review pipeline: pre-filter with SAST, enrich with LLM analysis
- Managing false positives: confidence scoring, auto-dismiss low-confidence findings
- Diff-aware review: analyzing only changed code rather than entire files

## Why This Matters

For teams shipping LLM applications, LLM-assisted review is both a force multiplier and a unique challenge: the same LLM that assists review could itself introduce vulnerabilities in generated code. Understanding how to use LLMs securely in the review loop — with appropriate skepticism and human oversight — is a core DevSecOps skill.


## Security Review Prompt Templates

In [ ]:
from dataclasses import dataclass
from typing import List, Dict, Optional
import json

SECURITY_REVIEW_SYSTEM_PROMPT = '''
You are a security code reviewer specializing in Python web applications and LLM-based systems.
Your task is to analyze code for security vulnerabilities.

For each finding, provide:
- vulnerability_type: OWASP category or CWE name
- severity: CRITICAL | HIGH | MEDIUM | LOW
- line_number: approximate line
- description: what the vulnerability is and how it could be exploited
- remediation: specific code fix or architectural change
- confidence: HIGH | MEDIUM | LOW (your confidence this is a real finding, not FP)

Focus on: SQL injection, XSS, SSRF, authentication flaws, authorization bypasses,
insecure deserialization, prompt injection surfaces, secrets in code, and path traversal.
Respond with a JSON array of findings only.
'''

@dataclass
class LLMReviewFinding:
    vulnerability_type: str
    severity: str
    line_number: int
    description: str
    remediation: str
    confidence: str
    source: str = 'llm'  # 'llm', 'sast', 'human'

def build_review_prompt(code: str, context: str = '') -> str:
    parts = []
    if context:
        parts.append(f'Context: {context}')
    parts.append(f'Review this code for security vulnerabilities:')
    parts.append(f'```python\n{code}\n```')
    return '\n\n'.join(parts)

# Simulate LLM response (what GPT-4/Claude would return)
def simulate_llm_review(code: str) -> List[LLMReviewFinding]:
    '''Simulated LLM review response for demonstration.'''
    # In production: call anthropic.Anthropic().messages.create() with structured output

    simulated_response = [
        {
            'vulnerability_type': 'SQL Injection (CWE-89)',
            'severity': 'CRITICAL',
            'line_number': 8,
            'description': 'String formatting used to build SQL query. '
                           'Attacker can inject arbitrary SQL by manipulating user_id.',
            'remediation': 'Use parameterized queries: cursor.execute("SELECT * FROM users WHERE id = %s", (user_id,))',
            'confidence': 'HIGH'
        },
        {
            'vulnerability_type': 'Prompt Injection Surface (CWE-77)',
            'severity': 'HIGH',
            'line_number': 15,
            'description': 'User input concatenated directly into LLM prompt. '
                           'Attacker can inject instructions to override system prompt.',
            'remediation': 'Sanitize user input, use structured message format, '
                           'consider input length limits and character allowlists.',
            'confidence': 'HIGH'
        },
        {
            'vulnerability_type': 'SSRF (CWE-918)',
            'severity': 'HIGH',
            'line_number': 22,
            'description': 'URL from user input passed to requests.get() without validation. '
                           'Attacker can probe internal services (169.254.169.254, 10.x.x.x).',
            'remediation': 'Validate URL against allowlist; block RFC1918 ranges.',
            'confidence': 'HIGH'
        },
        {
            'vulnerability_type': 'Debug Endpoint Exposure',
            'severity': 'MEDIUM',
            'line_number': 30,
            'description': 'debug=True in Flask/FastAPI exposes stack traces and debugger.',
            'remediation': 'Set debug=False in production; use env var APP_DEBUG.',
            'confidence': 'MEDIUM'
        },
    ]
    return [LLMReviewFinding(**f) for f in simulated_response]

SAMPLE_VULNERABLE_CODE = '''
from flask import Flask, request
import sqlite3
import requests

app = Flask(__name__, debug=True)

def get_user(user_id):
    conn = sqlite3.connect('app.db')
    query = f"SELECT * FROM users WHERE id = {user_id}"  # SQL injection
    return conn.execute(query).fetchone()

@app.post('/chat')
def chat():
    user_msg = request.json['message']
    prompt = f"System: helpful assistant. User: {user_msg}"  # prompt injection
    response = call_llm(prompt)
    return {'response': response}

@app.get('/fetch')
def fetch_url():
    url = request.args.get('url')  # SSRF
    return requests.get(url).text
'''

findings = simulate_llm_review(SAMPLE_VULNERABLE_CODE)
print(f'LLM review findings: {len(findings)}')
for f in findings:
    print(f'  [{f.severity}] L{f.line_number}: {f.vulnerability_type} (confidence: {f.confidence})')


## False Positive Management & Confidence Scoring

In [ ]:
def filter_findings(findings: List[LLMReviewFinding],
                     min_confidence: str = 'MEDIUM',
                     min_severity: str = 'MEDIUM') -> List[LLMReviewFinding]:
    CONFIDENCE_ORDER = ['LOW', 'MEDIUM', 'HIGH']
    SEVERITY_ORDER = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']

    min_conf_idx = CONFIDENCE_ORDER.index(min_confidence)
    min_sev_idx = SEVERITY_ORDER.index(min_severity)

    return [
        f for f in findings
        if CONFIDENCE_ORDER.index(f.confidence) >= min_conf_idx
        and SEVERITY_ORDER.index(f.severity) >= min_sev_idx
    ]

def merge_sast_and_llm(sast_findings: List[Dict],
                        llm_findings: List[LLMReviewFinding]) -> List[LLMReviewFinding]:
    '''Findings in both SAST and LLM get confidence bumped to HIGH.'''
    merged = list(llm_findings)

    for sast in sast_findings:
        matching_llm = [
            f for f in merged
            if abs(f.line_number - sast.get('line', 0)) <= 2
        ]
        if matching_llm:
            matching_llm[0].confidence = 'HIGH'
            matching_llm[0].description += ' [SAST-CONFIRMED]'
        else:
            merged.append(LLMReviewFinding(
                vulnerability_type=sast.get('type', 'Unknown'),
                severity=sast.get('severity', 'MEDIUM'),
                line_number=sast.get('line', 0),
                description=sast.get('description', ''),
                remediation=sast.get('fix', ''),
                confidence='HIGH',
                source='sast'
            ))
    return merged

# Simulate SAST findings for same file
sast_findings = [
    {'type': 'SQL Injection', 'severity': 'CRITICAL', 'line': 8,
     'description': 'String formatting in SQL query', 'fix': 'Use parameterized queries'},
]

merged = merge_sast_and_llm(sast_findings, findings)
actionable = filter_findings(merged, min_confidence='MEDIUM', min_severity='HIGH')

print(f'Total findings: {len(merged)}')
print(f'Actionable (HIGH+ severity, MEDIUM+ confidence): {len(actionable)}')
print()
for f in actionable:
    sast_confirmed = '[SAST-CONFIRMED]' in f.description
    print(f'  [{f.severity}] {f.vulnerability_type}')
    print(f'    Confidence: {f.confidence}{" (SAST+LLM)" if sast_confirmed else ""}')
    print(f'    Fix: {f.remediation[:80]}')
